# ToMe Reproduction — Figures

Loads `results/E*.json`, renders paper-style throughput-vs-accuracy curves (Fig 3a/b/c) and ablation/matching bar charts inline, and saves PNGs to `results/figures/`.

Measured on local GPU (fp32). Paper throughputs are V100 — absolute im/s differ by hardware; accuracy is directly comparable.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
RES = ROOT / "results"
OUT = RES / "figures"
OUT.mkdir(parents=True, exist_ok=True)

def load(name):
    p = RES / f"{name}.json"
    return json.load(open(p)) if p.exists() else None

e1 = load("E1_augreg")
e2 = load("E2_mae")
e3 = load("E3_swag")
e3h = load("E3_swag_h")
e4 = load("E4_ablation")
e4a = load("E4_ablation_augreg")
e5 = load("E5_matching")

if e3 and e3h:
    e3 = e3 + e3h
elif e3h and not e3:
    e3 = e3h

print({k: (len(v) if v else 0) for k, v in
       [("E1", e1), ("E2", e2), ("E3", e3), ("E4", e4), ("E4a", e4a), ("E5", e5)]})

## Paper reference values

In [ ]:
PAPER_E1_R0 = {"vit_tiny_patch16_224": 75.5, "vit_small_patch16_224": 81.4,
               "vit_base_patch16_224": 84.5, "vit_large_patch16_224": 85.8,
               "vit_large_patch16_384": 86.6}
PAPER_E2_R0 = {"mae_vit_base": 83.7, "mae_vit_large": 85.96, "mae_vit_huge": 86.9}
PAPER_T1 = {
    ("a", "x_pre"): (83.02, 186.8), ("a", "x"): (83.70, 182.8), ("a", "k"): (84.25, 182.9),
    ("a", "q"): (84.04, 182.8), ("a", "v"): (83.80, 182.9),
    ("b", "euclidean"): (84.26, 182.5), ("b", "cosine"): (84.25, 182.9),
    ("b", "dot"): (82.78, 183.0), ("b", "softmax"): (82.00, 183.0),
    ("c", "concat"): (84.32, 180.3), ("c", "mean"): (84.25, 182.9),
    ("d", "keep_one"): (81.01, 185.4), ("d", "max"): (83.50, 184.6),
    ("d", "avg"): (83.57, 183.8), ("d", "wavg"): (84.25, 182.9),
    ("e", "sequential"): (81.07, 183.0), ("e", "alternating"): (84.25, 182.9),
    ("e", "random"): (83.80, 181.7),
    ("f", "mae_no_prop"): (84.25, 182.9), ("f", "mae_prop"): (83.84, 180.9),
    ("f", "augreg_no_prop"): (82.15, 182.8), ("f", "augreg_prop"): (83.51, 180.8),
}
PAPER_T2 = {
    "prune_random": (79.22, 184.4), "prune_attn": (79.48, 183.8),
    "kmeans2": (80.19, 169.7), "kmeans5": (80.29, 147.5),
    "greedy": (84.36, 179.4), "bipartite": (84.25, 182.9),
}

def group_by_model(rows):
    g = defaultdict(list)
    for r in rows:
        g[r["model"]].append(r)
    for m in g:
        g[m].sort(key=lambda x: x["r"])
    return g

## Fig 3a/b/c — throughput vs accuracy sweep

In [ ]:
import math
import matplotlib.ticker as mticker

def plot_sweep(rows, title, fname, label_map=None, xticks=None):
    """throughput (log x) vs acc. Auto power-of-2 ticks covering data range."""
    g = group_by_model(rows)
    fig, ax = plt.subplots(figsize=(7, 5))
    all_thr = []
    for m, rs in g.items():
        xs = [r["throughput"] for r in rs]
        ys = [r["acc"] for r in rs]
        all_thr.extend(xs)
        lab = (label_map or {}).get(m, m)
        ax.plot(xs, ys, marker="o", ms=4, lw=1.5, label=lab)
        ax.annotate(f"r={rs[0]['r']}", (xs[0], ys[0]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
        ax.annotate(f"r={rs[-1]['r']}", (xs[-1], ys[-1]), fontsize=7,
                    xytext=(3, -8), textcoords="offset points")
    ax.set_xscale("log")
    if xticks is None:
        lo = math.floor(math.log2(min(all_thr)))
        hi = math.ceil(math.log2(max(all_thr)))
        xticks = [2 ** k for k in range(lo, hi + 1)]
    ax.set_xticks(xticks)
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    ax.set_xlim(xticks[0], xticks[-1])
    ax.set_xlabel("throughput (im/s, fp32)  [log scale]")
    ax.set_ylabel("ImageNet-1k top-1 acc (%)")
    ax.set_title(title)
    ax.legend(fontsize=9, loc="lower left")
    ax.grid(True, which="major", alpha=0.3)
    ax.grid(True, which="minor", alpha=0.12)
    fig.tight_layout()
    fig.savefig(OUT / fname, dpi=150)
    plt.show()
    print(f"saved {OUT/fname}")

AUG_LABELS = {"vit_tiny_patch16_224": "ViT-Ti/16", "vit_small_patch16_224": "ViT-S/16",
              "vit_base_patch16_224": "ViT-B/16", "vit_large_patch16_224": "ViT-L/16",
              "vit_large_patch16_384": "ViT-L/16@384"}
MAE_LABELS = {"mae_vit_base": "ViT-B/16", "mae_vit_large": "ViT-L/16",
              "mae_vit_huge": "ViT-H/14"}

if e1:
    plot_sweep(e1, "Fig 3a — AugReg ViT off-the-shelf", "fig3a_augreg.png", AUG_LABELS)
if e2:
    plot_sweep(e2, "Fig 3c — MAE ViT off-the-shelf", "fig3c_mae.png", MAE_LABELS)
if e3:
    plot_sweep(e3, "Fig 3b — SWAG ViT off-the-shelf", "fig3b_swag.png")
else:
    print("E3 not run yet — skipping Fig 3b")

## Table 1 — Ablation: measured vs paper (bar chart)

In [ ]:
if e4:
    merged = [r for r in e4 if r["backbone"] != "augreg"] + (e4a or [])
    merged.sort(key=lambda r: (r["sub_table"], r["label"]))
    labels = [f"{r['sub_table']}:{r['label']}" for r in merged]
    meas = [r["acc"] for r in merged]
    pap = [PAPER_T1.get((r["sub_table"], r["label"]), (None,))[0] for r in merged]

    import numpy as np
    x = np.arange(len(labels)); w = 0.4
    plt.figure(figsize=(13, 5))
    plt.bar(x - w/2, meas, w, label="measured")
    plt.bar(x + w/2, [p if p else 0 for p in pap], w, label="paper")
    plt.xticks(x, labels, rotation=75, ha="right", fontsize=8)
    plt.ylabel("top-1 acc (%)"); plt.ylim(78, 86)
    plt.title("Table 1 — Ablation (ViT-L/16 MAE, r=8): measured vs paper")
    plt.legend(); plt.grid(True, axis="y", alpha=0.3); plt.tight_layout()
    plt.savefig(OUT / "table1_ablation.png", dpi=150)
    plt.show()
    print(f"saved {OUT/'table1_ablation.png'}")
else:
    print("E4 not loaded")

## Table 2 — Matching: measured vs paper (bar chart)

In [ ]:
if e5:
    order = ["prune_random", "prune_attn", "kmeans2", "kmeans5", "greedy", "bipartite"]
    idx = {r["algo"]: r for r in e5}
    algos = [a for a in order if a in idx]
    meas = [idx[a]["acc"] for a in algos]
    pap = [PAPER_T2[a][0] for a in algos]

    import numpy as np
    x = np.arange(len(algos)); w = 0.4
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.bar(x - w/2, meas, w, label="measured")
    ax1.bar(x + w/2, pap, w, label="paper")
    ax1.set_xticks(x); ax1.set_xticklabels(algos, rotation=30, ha="right")
    ax1.set_ylabel("top-1 acc (%)"); ax1.set_ylim(74, 86)
    ax1.set_title("acc: measured vs paper"); ax1.legend(); ax1.grid(True, axis="y", alpha=0.3)

    meas_t = [idx[a]["throughput"] for a in algos]
    pap_t = [PAPER_T2[a][1] for a in algos]
    ax2.bar(x - w/2, meas_t, w, label="measured (local)")
    ax2.bar(x + w/2, pap_t, w, label="paper (V100)")
    ax2.set_xticks(x); ax2.set_xticklabels(algos, rotation=30, ha="right")
    ax2.set_ylabel("throughput (im/s)")
    ax2.set_title("throughput: local vs V100"); ax2.legend(); ax2.grid(True, axis="y", alpha=0.3)
    plt.suptitle("Table 2 — Matching algorithms (ViT-L/16 MAE, r=8)")
    plt.tight_layout()
    plt.savefig(OUT / "table2_matching.png", dpi=150)
    plt.show()
    print(f"saved {OUT/'table2_matching.png'}")
else:
    print("E5 not loaded")